# D2.8 · Regulatory clock

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *Security of AI*

Builds on **[D2.7 · Stop authority](https://spbreed.github.io/cyber-commons/lessons/D2.7.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Run the first-hour checklist in a tabletop.

**Why a security engineer needs it.** Notification obligations discovered in week two. The control it builds is: feed Track E2 in hour one.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The disclosure clock starts on the incident, not on your understanding of it. Materiality for a probabilistic actor is genuinely hard, and the hard part does not pause the clock.

> **At CyberTravels.** The disclosure clock started when CyberTravels exported the customer profiles, not when CyberTravels understood what had happened. Passport and payment data make the deadline short. R10.

## 2 · The framework

```
   incident starts -------------------------------> deadline
        |                |                |
     detected        understood        reportable?
                          ^
              materiality for a probabilistic actor is genuinely hard
              and the difficulty does not pause the clock

   trigger criteria are written before, or they are written badly
```

Regulatory clocks start at **awareness** — the point at which you know a
reportable event may have occurred. Not at confirmation, not at containment.

Two consequences that teams discover on day three:

1. **Containing fast does not buy reporting time.** You can contain in an hour
   and still miss a 72-hour deadline, because the clock never paused.
2. **Broken attribution consumes the clock.** If you cannot say who acted
   (D2.1), scoping takes days, and those days are deadline days.

Containment and disclosure are separate workstreams competing for the same
people. If your runbook has one owner for both, one of them is being done badly
under time pressure.

## 3 · Demo — the clock under four scenarios

In [ ]:
import time

def clock(awareness, containment, report, deadline_hours=72):
    to_contain = (containment - awareness) / 3600
    to_report  = (report - awareness) / 3600
    return {"hours_to_containment": round(to_contain, 1),
            "hours_to_report": round(to_report, 1),
            "deadline": deadline_hours,
            "met": to_report <= deadline_hours,
            "margin": round(deadline_hours - to_report, 1)}

t0 = time.time()
H = 3600
SCENARIOS = {
 "fast containment, slow scoping": (t0 + 1*H,  t0 + 80*H),
 "slow containment, fast reporting": (t0 + 40*H, t0 + 60*H),
 "both fast":                      (t0 + 2*H,  t0 + 20*H),
 "attribution broken (D2.1)":      (t0 + 6*H,  t0 + 92*H),
}
print(f"{'scenario':34s}{'contain':>9}{'report':>9}{'met':>6}{'margin':>9}")
print("-" * 68)
for name, (c, r) in SCENARIOS.items():
    k = clock(t0, c, r)
    print(f"{name:34s}{k['hours_to_containment']:>9.1f}{k['hours_to_report']:>9.1f}"
          f"{str(k['met']):>6}{k['margin']:>9.1f}")
print("\nThe first row contained in ONE HOUR and still missed the deadline.")

## 4 · Where it breaks — the clock starts earlier than people think

In [ ]:
TIMELINE = [
 ("alert fires",                          0,  False),
 ("analyst triages, suspects an incident", 3, True),   # ← awareness, arguably
 ("IR lead confirms an incident",         9,  True),
 ("scope established",                    40, True),
 ("legal confirms it is reportable",      55, True),
]
print(f"{'event':40s}{'t+h':>6}  could a regulator call this awareness?")
print("-" * 84)
for name, h, aware in TIMELINE:
    print(f"{name:40s}{h:>6}  {aware}")

report_at = 76
for label, start_h in (("clock from analyst suspicion", 3),
                       ("clock from IR confirmation", 9),
                       ("clock from legal determination", 55)):
    hours = report_at - start_h
    print(f"\n{label:34s} elapsed {hours:>3}h  "
          f"{'MET' if hours <= 72 else 'MISSED'} (72h deadline)")
print("\nThe same incident, the same report time, three different answers.")
print("Pick the earliest defensible start. A regulator will.")

## 5 · The control — separate owners, and a shortest-clock register

In [ ]:
OBLIGATIONS = {
 "GDPR (personal data breach)":     (72,  "supervisory authority"),
 "DORA (major ICT incident)":       (4,   "initial notification"),
 "NIS2 (early warning)":            (24,  "CSIRT"),
 "PCI DSS (card data)":             (24,  "acquirer/brands"),
 "contractual (major client)":      (12,  "client security contact"),
}
print(f"{'obligation':36s}{'deadline (h)':>14}  notify")
print("-" * 76)
for name, (hours, who) in sorted(OBLIGATIONS.items(), key=lambda kv: kv[1][0]):
    print(f"{name:36s}{hours:>14}  {who}")
shortest = min(OBLIGATIONS.items(), key=lambda kv: kv[1][0])
print(f"\nyour real deadline is the shortest: {shortest[0]} at {shortest[1][0]}h")

def runbook_check(containment_owner, disclosure_owner, clock_starts_at):
    problems = []
    if containment_owner == disclosure_owner:
        problems.append("one owner for both workstreams — they compete for the "
                        "same person under time pressure")
    if clock_starts_at != "awareness":
        problems.append(f"clock starts at {clock_starts_at!r}, not at awareness — "
                        f"a regulator will use the earlier point")
    return (not problems), problems

for label, args in (("as usually written", ("IR lead", "IR lead", "confirmation")),
                    ("corrected", ("IR lead", "legal/compliance lead", "awareness"))):
    ok, problems = runbook_check(*args)
    print(f"\n{label}: sound={ok}")
    for p in problems: print(f"   ⚠ {p}")
assert runbook_check("IR lead", "legal/compliance lead", "awareness")[0]

## What you just proved

One-hour containment still misses the 72-hour deadline in the slow-scoping scenario, and broken attribution misses it by 20 hours. The same incident is met or missed depending on which point is treated as awareness. The obligation register shows DORA's 4-hour clock as the binding one, and the runbook check flags a shared owner and a late clock start.

## Your turn

Build your shortest-clock register: every obligation, its deadline, and who notifies. Then check whether your runbook starts the clock at awareness or at confirmation. The gap between those two is often more than a day.

---

**Next → [D2.9 · The fleet kill switch](https://spbreed.github.io/cyber-commons/lessons/D2.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*